In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def process_sensor_data(file_path, label_name, window_size):
    # Load the data
    df = pd.read_csv(file_path)
    
    # Calculate magnitude: sqrt(ax^2 + ay^2 + az^2)
    df['magnitude'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)
    
    # Start relative time at zero in seconds (from timestamp_ms)
    df['relative_time'] = (df['timestamp_ms'] - df['timestamp_ms'].iloc[0]) / 1000.0
    
    # Calculate the total number of complete windows
    max_time = df['relative_time'].max()
    num_windows = int(np.floor(max_time / window_size))
    
    results = []
    
    for i in range(num_windows):
        start_t = i * window_size
        end_t = (i + 1) * window_size
        
        # Get data within this window
        mask = (df['relative_time'] >= start_t) & (df['relative_time'] < end_t)
        window_data = df[mask]
        
        # Calculate mean and population standard deviation (ddof=0)
        mean_mag = window_data['magnitude'].mean()
        std_mag = window_data['magnitude'].std(ddof=0)
        
        results.append({
            'start_s': start_t,
            'end_s': end_t,
            'mean_mag': mean_mag,
            'std_mag': std_mag,
            'label': label_name
        })
        
    features_df = pd.DataFrame(results)
    
    return df, features_df

In [ ]:
# Define window size (change to 1 for initial test, 2 for final export)
window_size = 2

# Process both files
still_raw, still_features = process_sensor_data('LAB1/accel_standing_still_20260922_105606.csv', 'still', window_size)
walk_raw, walk_features = process_sensor_data('LAB1/accel_walking_20260922_110005.csv', 'walk', window_size)

# Display window counts
print(f"Window Size: {window_size} seconds")
print(f"Still recording: {len(still_features)} windows")
print(f"Walk recording: {len(walk_features)} windows")

In [ ]:
# Magnitude-versus-time plots
plt.figure(figsize=(14, 5))

# Still Plot
plt.subplot(1, 2, 1)
plt.plot(still_raw['relative_time'], still_raw['magnitude'], label='Still', color='blue', alpha=0.7)
plt.title(f'Still Activity - Magnitude vs Time (Window: {window_size}s)')
plt.xlabel('Time (s)')
plt.ylabel('Magnitude (m/s^2)')
plt.grid(True)
plt.legend()

# Walk Plot
plt.subplot(1, 2, 2)
plt.plot(walk_raw['relative_time'], walk_raw['magnitude'], label='Walk', color='orange', alpha=0.7)
plt.title(f'Walk Activity - Magnitude vs Time (Window: {window_size}s)')
plt.xlabel('Time (s)')
plt.ylabel('Magnitude (m/s^2)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# Scatter plot of mean magnitude versus standard deviation
plt.figure(figsize=(8, 6))
plt.scatter(still_features['mean_mag'], still_features['std_mag'], label='Still', alpha=0.7, color='blue')
plt.scatter(walk_features['mean_mag'], walk_features['std_mag'], label='Walk', alpha=0.7, color='orange')
plt.title(f'Feature Scatter Plot (Window: {window_size}s)')
plt.xlabel('Mean Magnitude (m/s^2)')
plt.ylabel('Standard Deviation (m/s^2)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Export to CSV without index
if window_size == 2:
    expected_columns = ['start_s', 'end_s', 'mean_mag', 'std_mag', 'label']
    assert list(still_features.columns) == expected_columns, "Still features columns do not match."
    assert list(walk_features.columns) == expected_columns, "Walk features columns do not match."
    
    still_features.to_csv('still_features.csv', index=False)
    walk_features.to_csv('walk_features.csv', index=False)
    print("Successfully exported still_features.csv and walk_features.csv.")
else:
    print(f"Current window_size is {window_size}. Set window_size = 2 to export.")